In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install necessary libraries (Hugging Face ecosystem, LangChain, PEFT, CUDA utilities, Indic transliteration)
!pip install -q transformers==4.37.2 peft==0.9.0 accelerate bitsandbytes sentence-transformers faiss-cpu pandas numpy indic-transliteration transformers_stream_generator
!pip install -q -U langchain==0.3.27 langchain-core==0.3.79 langchain-community==0.3.31

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.6/159.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.8/449.8 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from peft import PeftModel
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_core.documents import Document

# Transliteration Imports (already available, kept for completeness)
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

# --- Configuration ---
# Update paths to point to your project folder inside Google Drive
PROJECT_DIR = "/content/drive/MyDrive/sanskrity"
DATA_PATH = os.path.join(PROJECT_DIR, "res/combined.csv")
ADAPTER_PATH = os.path.join(PROJECT_DIR, "qwen-finetuned")
INDEX_PATH = os.path.join(PROJECT_DIR, "sanskrit_faiss_index_itihasa")
BASE_MODEL_NAME = "Qwen/Qwen-1_8B" # The base model you used for fine-tuning
os.makedirs(os.path.join(PROJECT_DIR, "res"), exist_ok=True) # Ensure directories exist

# Check for CUDA availability
if torch.cuda.is_available():
    print(f"CUDA is available. Device: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available. This script may run very slowly.")

# --- 0. Transliteration Function ---
def devanagari_to_roman(devanagari_text, output_scheme=sanscript.IAST):
    """Transliterates a Devanagari string to a Roman scheme (defaulting to IAST)."""
    try:
        romanized = transliterate(devanagari_text, sanscript.DEVANAGARI, output_scheme)
        return romanized
    except Exception as e:
        return f"Error during transliteration: {e}"

CUDA is available. Device: NVIDIA A100-SXM4-80GB


In [ ]:
def load_and_chunk_itihasa_data(csv_path):
    """
    Loads data from the CSV and creates LangChain Documents using only
    the English translation for robust embedding.
    """
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"ERROR: CSV file not found at {csv_path}. Please ensure your file is in the drive.")
        # Create an empty DataFrame for graceful failure
        df = pd.DataFrame(columns=['sn', 'en', 'book', 'chapter', 'shloka'])

    documents = []
    for index, row in df.iterrows():
        content = row['sn']

        # Store citation data in metadata
        metadata = {
            "book": row['book'],
            "chapter": row['chapter'],
            "shloka_num": row['shloka'],
            "english_text": row['en']
        }
        documents.append(Document(page_content=content, metadata=metadata))

    return documents

# Load and process the data
chunks = load_and_chunk_itihasa_data(DATA_PATH)
if not chunks:
    print("FATAL: No documents loaded. Cannot proceed with RAG setup.")
else:
    print(f"Loaded and processed {len(chunks)} shloka Documents.")

Loaded and processed 93030 shloka Documents.


In [64]:
# embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

if os.path.exists(INDEX_PATH):
    # Load the existing index
    print(f"Loading existing FAISS index from disk: {INDEX_PATH}")
    vector_store = FAISS.load_local(
        folder_path=INDEX_PATH,
        embeddings=embedding_model,
        allow_dangerous_deserialization=True
    )
else:
    # If the index doesn't exist, create and save it
    print("FAISS index not found. Creating and saving new index...")
    if chunks:
        vector_store = FAISS.from_documents(chunks, embedding_model)
        vector_store.save_local(INDEX_PATH)
        print("New FAISS index saved to disk.")
    else:
        print("Cannot create FAISS index: No documents found.")
        # Placeholder vector_store to prevent script crash
        vector_store = FAISS.from_documents([Document(page_content="Placeholder")], embedding_model)

retriever = vector_store.as_retriever(k=1)

FAISS index not found. Creating and saving new index...
New FAISS index saved to disk.


In [53]:
#Setup Local Fine-Tuned LLM
def setup_local_llm(base_model_name, adapter_path):
    """
    Initializes and wraps the local fine-tuned Qwen model (LoRA adapter)
    in a LangChain pipeline, ensuring correct loading of the PEFT weights.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    try:
        # 1. Load Tokenizer (Use base_model_name as adapter path usually lacks tokenizer files)
        tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

        # 2. Load the Base Model
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )

        # 3. Load LoRA Adapter onto the Base Model (CRITICAL STEP)
        model = PeftModel.from_pretrained(base_model, adapter_path)
        model = model.merge_and_unload() # Optional: Merge weights for potential speedup
        model.eval()

        print(f"Successfully loaded fine-tuned model: {base_model_name} with adapter from {adapter_path}")

    except Exception as e:
        print(f"Error loading fine-tuned model: {e}")
        print("Using base Qwen model for structural testing (will likely be poor quality).")
        # Fallback to base model for structural testing if LoRA loading fails
        tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="auto", trust_remote_code=True)

    # --- Setup the HuggingFace Pipeline ---
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512, # Increased tokens for detailed answers
        temperature=0.02, # Low temperature for factual RAG
        do_sample=True,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        # IMPORTANT: return_full_text=False is key for LangChain
        return_full_text=False
    )

    llm = HuggingFacePipeline(pipeline=pipe)
    return llm

llm = setup_local_llm(BASE_MODEL_NAME, ADAPTER_PATH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Error loading fine-tuned model: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.
Using base Qwen model for structural testing (will likely be poor quality).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [61]:
#
QA_PROMPT = PromptTemplate(
    template="""### Human:
CONTEXT:
{context}

Based *only* on the CONTEXT, analyze the most probable verse cited as {context_citation_1}.

QUESTION: {question}

### Assistant:
The most relevant verse is **{context_citation_1}**.

**Sanskrit Text:** {sanskrit_1}

**English Translation:** {english_1}

**Explanation:** [Explain the meaning based on the translation and context, focusing on the first shloka's context and meaning.]
""",
    input_variables=["context", "question", "context_citation_1", "sanskrit_1", "english_1"],
)

# Create the RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": QA_PROMPT},
    return_source_documents=True # Allows us to debug the retrieved documents
)

print("RAG Chain Initialized with Fine-Tuned Qwen Model.")

RAG Chain Initialized with Fine-Tuned Qwen Model.


In [65]:
def chat_with_assistant():
    print("\n--- Sanskrit Scripture Assistant Initialized ---")
    print("Ask a question about the scriptures (e.g., 'What is the nature of the self according to Gita?') or type 'quit' to exit.")

    #QA_PROMPT template
    global QA_PROMPT

    while True:
        user_query = input("\nYour Question: ")
        if user_query.lower() in ["quit", "exit"]:
            print("Exiting assistant.")
            break

        retrieved_docs = retriever.get_relevant_documents(user_query)

        # 3. CUSTOM CONTEXT STRING CREATION
        custom_context_parts = []
        first_doc = retrieved_docs[0]
        first_doc_meta = first_doc.metadata
        context_citation_1 = f"{first_doc_meta.get('book')} {first_doc_meta.get('chapter')}.{first_doc_meta.get('shloka_num')}"
        sanskrit_text_1 = first_doc.page_content
        english_text_1 = first_doc_meta.get("english_text", "ERROR: English translation not found in metadata.")

        for i, doc in enumerate(retrieved_docs):
            meta = doc.metadata
            sanskrit_text = doc.page_content
            english_text = meta.get("english_text", "ERROR: English translation not found in metadata.")
            source_citation = f"{meta.get('book')} {meta.get('chapter')}.{meta.get('shloka_num')}"

            # Format the context block
            context_block = f"""
--- SHLOKA {i+1} (Source: {source_citation}) ---
Sanskrit: {sanskrit_text}
English Translation: {english_text}
"""
            custom_context_parts.append(context_block)

        custom_context = "\n".join(custom_context_parts)

        # 5. MANUAL LLM INVOCATION
        final_prompt = QA_PROMPT.format(
            context=custom_context,
            question=user_query,
            context_citation_1=context_citation_1, # Citation for the header
            sanskrit_1=sanskrit_text_1,           # Full Sanskrit text injection
            english_1=english_text_1              # Full English text injection
        )

        # Invoke the LLM directly
        result_text = llm.invoke(final_prompt)

        # Invoke the LLM directly
        result_text = llm.invoke(final_prompt)

        # We reformat the final result to match the structure of the RetrievalQA output
        result = {'result': result_text, 'source_documents': retrieved_docs}

        # --- Debugging Output (Kept for verification) ---
        print("\nRetrieved")
        if 'source_documents' in result:
            for i, doc in enumerate(result['source_documents']):
                meta = doc.metadata
                print(f"[{i+1}] Source: {meta.get('book')} {meta.get('chapter')}.{meta.get('shloka_num')}")
                print(f" Shloka: {doc.page_content[:100]}...")
                print(f" English Content: {meta.get('english_text', 'N/A')[:250]}...\n")

        # 4b. Final Answer (LLM's response)
        print(f"\n Answer:\n{result['result']}")


if __name__ == "__main__":
    chat_with_assistant()


--- Sanskrit Scripture Assistant Initialized ---
Ask a question about the scriptures (e.g., 'What is the nature of the self according to Gita?') or type 'quit' to exit.

Your Question: तस्यां पुर्यामयोध्यायां वेदवित्सर्वसंग्रहः | दीर्घदर्शी महातेजाः पौरजानपदप्रियः

Retrieved
[1] Source: ramayana 6.1
 Shloka (Sanskrit): तस्यां पुर्यामयोध्यायां वेदवित् सर्वसंग्रहः। दीर्घदर्शी महातेजाः पौरजानपदप्रियः॥...
 English Content (Metadata): In that city of Ayodhyā resided king Dasaratha, versed in the Vedas, commanding all resources, far-sighted, of mighty prowess, dear to the inhabitants both rural and urban....

[2] Source: mahabharata 70.12
 Shloka (Sanskrit): दीर्घदर्शी महातेजाः पौरजानपदप्रियः। पाण्डवानामतिरथो यज्ञधर्मपरो वशी॥...
 English Content (Metadata): He is far-sighted, highly energetic, beloved of the citizens and villagers, the foremost of car-warriors among the sons of Pandu, the performer of sacrifices, pious, and selfcontrolled....

[3] Source: mahabharata 63.2
 Shloka (Sanskrit)

/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(



Retrieved
[1] Source: ramayana 55.1
 Shloka (Sanskrit): ततस्तानाकुलान् दृष्ट्वा विश्वामित्रास्त्रमोहितान्। वसिष्ठश्चोदयामास कामधुक् सृज योगतः॥...
 English Content (Metadata): And beholding them sore harassed, and overwhelmed by Visvāmitra's weapons, Vasistha directed (Sabala) saying, 'o cow of plenty, do you create (fresh troops), through your Yoga power.'...

[2] Source: mahabharata 34.5
 Shloka (Sanskrit): अक्षांश्च दृष्ट्वा शकुनेर्यथावत् कामानुकूलानयुजो युजश्च। शक्यो नियन्तुमभविष्यदात्मा मन्युस्तु हन्यात...
 English Content (Metadata): Seeing the dice obedient to the wishes of Shakuni in Ayuja and Yuja (odds and evens), I could have controlled my mind, but anger (you know) drives off a person's patience....

[3] Source: ramayana 52.1
 Shloka (Sanskrit): तं दृष्ट्वा परमप्रीतो विश्वामित्रो महाबलः। प्रणतो विनयाद् वीरो वसिष्ठं जपतां वरम्॥...
 English Content (Metadata): Upon seeing that foremost of these reciting mantras, the highly powerful and heroic Viśvāmitra, exceedingly delighted